In [ ]:
import torch
import torch.nn as nn
from nltk.stem.porter import PorterStemmer
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
import nltk
nltk.download('stopwords')
stopwords.words('english')

## Loading fakenews data from kaggle

In [ ]:
import kagglehub
import pandas as pd
import os
path = kagglehub.dataset_download("algord/fake-news")

for dirname, _, filenames in os.walk('/kaggle/input'):
    print(dirname)
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:

df=pd.read_csv(f'{path}/FakeNewsNet.csv')
df.head()

In [ ]:
df=df[['title','source_domain','real']]
df.head()

In [ ]:
print(f'Duplicate Titles: {df.duplicated(['title']).sum()} \n Na values {df.isna().sum()}')

df.dropna(inplace=True)

df.drop_duplicates( inplace=True,subset=['title'],keep='first')

# print(df.shape)
df.info()

## Processing the text

In [ ]:
import re
stemmer=PorterStemmer()

def process_txt(content):
    content=re.sub(r'[^a-zA-Z]', ' ', str(content)).lower().split()
    return ' '.join([stemmer.stem(word) for word in content if word  not in stopwords.words("english")])
# process_txt(df['title'][0]) 
vectorizor=TfidfVectorizer()
df['title']=df['title'].apply(process_txt)
df.head()

In [ ]:
from sklearn.model_selection import train_test_split
X=vectorizor.fit_transform(df['title'])
Y = df["real"].astype(int).to_numpy()
x_train, x_test,y_train,y_test = train_test_split(X,Y)
x_train.shape

In [ ]:
x_train_t = torch.tensor(x_train.toarray(), dtype=torch.float32)
x_test_t = torch.tensor(x_test.toarray(), dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
y_test_t = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

In [ ]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.model=nn.Sequential(
            nn.Linear(x_train_t.shape[1],256), #This selects size of a single row
            nn.ReLU(),
            nn.Linear(256,128),
            nn.ReLU(),
            nn.Linear(128,64),
            nn.ReLU(),
            nn.Linear(64,1)
        )
    def forward(self,x):
        return self.model(x)

In [ ]:
import torch.optim as optim
model =NeuralNetwork()
optimizer=optim.Adam(model.parameters(),lr=0.01)
criterion=nn.BCEWithLogitsLoss()


##  Training


In [ ]:
epochs=12
for epoch in range(epochs):
    out=model(x_train_t)
    loss=criterion(out,y_train_t)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    print(f'epoch {epoch}: Loss: {loss.item()}')
